# 01 — Sanity checks on the ingested data

Quick checks on `data/processed/*.parquet` (produced by `f1pred-ingest`):
coverage, pole → win conversion by regulation era, and 2026 winners so far.

Two definitions of *pole*: **grid = 1** (actual starting position, after penalties) and **qualifying P1**. Races with an unpublished grid are excluded from the grid-based rate.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path('..') / 'data' / 'processed'
t = {p.stem: pd.read_parquet(p) for p in DATA.glob('*.parquet')}
report = json.loads((DATA / 'ingest_report.json').read_text(encoding='utf-8'))
print('generated_at:', report['generated_at'], '| latest round:', report['latest_round'])
pd.Series(report['row_counts'], name='rows').to_frame()

## Coverage per season

In [ ]:
races, res, q = t['races'], t['results'], t['qualifying']
cov = pd.DataFrame({
    'scheduled': races.groupby('season').size(),
    'with_results': res.groupby('season')['round'].nunique(),
    'with_quali': q.groupby('season')['round'].nunique(),
    'sprints': t['sprint_results'].groupby('season')['round'].nunique()
    if not t['sprint_results'].empty else 0,
    'standings_rounds': t['driver_standings'].groupby('season')['round'].nunique(),
}).fillna(0).astype(int)
cov

## Pole → win conversion by era

In [ ]:
def era(season):
    return '2014–2021' if season <= 2021 else ('2022–2025' if season <= 2025 else '2026')

winners = res[res['position'] == 1][['season', 'round', 'driver_id', 'constructor_id', 'grid']]
qpole = q[q['position'] == 1][['season', 'round', 'driver_id']].rename(
    columns={'driver_id': 'quali_pole'})
w = winners.merge(qpole, on=['season', 'round'], how='left')
w['era'] = w['season'].map(era)
w['won_from_grid1'] = (w['grid'] == 1).where(w['grid'].notna())
w['won_from_quali_pole'] = (w['driver_id'] == w['quali_pole']).where(w['quali_pole'].notna())

summary = w.groupby('era').agg(
    races=('round', 'size'),
    races_with_grid=('won_from_grid1', 'count'),
    pole_win_pct_grid=('won_from_grid1', 'mean'),
    pole_win_pct_quali=('won_from_quali_pole', 'mean'),
)
summary[['pole_win_pct_grid', 'pole_win_pct_quali']] *= 100
summary.round(1)

In [ ]:
per_season = w.groupby('season')['won_from_grid1'].mean() * 100
colors = ['#7a8b99' if s <= 2021 else '#3b7dd8' if s <= 2025 else '#e8590c'
          for s in per_season.index]
ax = per_season.plot.bar(color=colors, figsize=(10, 3.5))
ax.set_ylabel('% of races won from grid P1')
ax.set_xlabel('')
ax.set_title('Pole-to-win conversion per season (grey 2014–21, blue 2022–25, orange 2026)')
ax.set_ylim(0, 100)
plt.tight_layout()

## 2026 winners so far

In [ ]:
d = t['drivers'].set_index('driver_id')
c = t['constructors'].set_index('constructor_id')
w26 = (w[w['season'] == 2026]
       .merge(races[['season', 'round', 'race_name', 'race_date']], on=['season', 'round'])
       .assign(winner=lambda x: x['driver_id'].map(d['family_name']),
               team=lambda x: x['constructor_id'].map(c['name']))
       [['round', 'race_name', 'race_date', 'winner', 'team', 'grid', 'quali_pole']])
w26

In [ ]:
pd.concat({'wins_by_driver': w26['winner'].value_counts(),
           'wins_by_team': w26['team'].value_counts()}, axis=1).fillna(0).astype(int)

## Gaps reported by the ingestion

In [ ]:
gaps = pd.DataFrame(report['gaps'])
gaps if not gaps.empty else 'no gaps reported'